# Laboratory 3 — Hebbian Association and STDP

Companion to Chapter 4. This controlled pairing example uses dimensionless activities and supplied output patterns. It demonstrates partial-cue recall and ambiguity, not autonomous learning. The STDP window is illustrated separately and does not train the association matrix.

[Run in Google Colab](https://colab.research.google.com/github/an-hongyu/Introduction-to-Neuromorphic-Embodied-AI/blob/main/notebooks/hebbian_learning.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
patterns = np.array([[1., 1., 0.], [0., 1., 1.]])
paired_outputs = np.eye(2)
weights = np.zeros((2, 3))
for _ in range(8):
    for x, y in zip(patterns, paired_outputs):
        weights += 0.1 * np.outer(y, x)
cues = np.array([[1., 0., 0.], [0., 0., 1.], [0., 1., 0.]])
scores = cues @ weights.T
assert np.allclose(weights, [[0.8, 0.8, 0], [0, 0.8, 0.8]])
assert np.allclose(scores, [[0.8, 0], [0, 0.8], [0.8, 0.8]])
print('Weights (output rows, input columns):\n', weights)
print('Partial-cue output scores:\n', scores)
print('The shared-feature cue is ambiguous; these are not probabilities.')

def pair_stdp(delta_ms, a_plus=0.01, a_minus=0.012, tau_ms=20.0):
    delta_ms = np.asarray(delta_ms, dtype=float)
    return np.where(delta_ms > 0, a_plus * np.exp(-delta_ms / tau_ms),
                    np.where(delta_ms < 0, -a_minus * np.exp(delta_ms / tau_ms), 0.0))
assert np.isclose(pair_stdp(10), 0.01 * np.exp(-0.5))
assert pair_stdp(-10) < 0 and pair_stdp(0) == 0

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
im = axes[0].imshow(weights, vmin=0, vmax=0.8, cmap='Blues')
axes[0].set(xticks=[0, 1, 2], xticklabels=['Feature 1', 'Shared', 'Feature 3'],
            yticks=[0, 1], yticklabels=['Output 1', 'Output 2'], title='A. Paired-pattern weights')
for row in range(2):
    for col in range(3):
        axes[0].text(col, row, '%.1f' % weights[row, col], ha='center', va='center',
                     color='white' if weights[row, col] > 0.4 else '#17334c')
fig.colorbar(im, ax=axes[0], shrink=0.7, label='Weight (a.u.)')
x = np.arange(3)
axes[1].bar(x - 0.18, scores[:, 0], 0.36, label='Output 1', color='#24628a')
axes[1].bar(x + 0.18, scores[:, 1], 0.36, label='Output 2', color='#d08a35')
axes[1].set(xticks=x, xticklabels=['Feature 1', 'Feature 3', 'Shared'], ylim=(0, 1),
            ylabel='Output score (a.u.)', title='B. Partial-cue recall')
axes[1].legend(fontsize=8)
delta = np.linspace(-80, 80, 801)
for mask, color in [(delta < 0, '#b6483b'), (delta > 0, '#268471')]:
    axes[2].plot(delta[mask], pair_stdp(delta[mask]), color=color)
axes[2].scatter([0], [0], color='0.3', s=15, zorder=3)
axes[2].axhline(0, color='0.6', lw=0.7)
axes[2].axvline(0, color='0.6', lw=0.7)
axes[2].set(xlabel='Post minus pre spike time (ms)', ylabel='Weight change (a.u.)',
            title='C. Illustrative STDP window')
plt.show()

## Experiments

1. Compare scores before and after pairing.
2. Shuffle the output pairings and compare recall.
3. Add overlapping patterns and measure interference.
4. Change the STDP time constant and inspect the timing window.

## References

[1] Donald O. Hebb, *The Organization of Behavior: A Neuropsychological Theory*. New York, NY, USA: Wiley, 1949. [Catalog record](https://wellcomecollection.org/works/g4sfzum4).

[2] Guo-qiang Bi and Mu-ming Poo, “Synaptic modifications in cultured hippocampal neurons: Dependence on spike timing, synaptic strength, and postsynaptic cell type,” *The Journal of Neuroscience*, vol. 18, no. 24, pp. 10464–10472, 1998, doi: [10.1523/JNEUROSCI.18-24-10464.1998](https://pubmed.ncbi.nlm.nih.gov/9852584/).